# ALDIMI - Merge salud y stock

Este notebook prepara el dataset de stock con la estructura requerida y lo vincula con el dataset de salud.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

In [ ]:
DATA_DIR = Path('data')
RAW_DIR = DATA_DIR / 'raw'
PROCESSED_DIR = DATA_DIR / 'processed'
MERGED_DIR = DATA_DIR / 'merged'

for d in (RAW_DIR, PROCESSED_DIR, MERGED_DIR):
    d.mkdir(parents=True, exist_ok=True)

health_df = pd.read_csv(RAW_DIR / 'health_raw.csv')
stock_raw = pd.read_csv(RAW_DIR / 'stock_raw.csv')

print('Health shape:', health_df.shape)
print('Stock shape:', stock_raw.shape)

In [ ]:
def find_col(df, keywords):
    for col in df.columns:
        low = col.lower()
        if any(k in low for k in keywords):
            return col
    return None

def ensure_fecha(df):
    df = df.copy()
    date_col = find_col(df, ['date', 'fecha', 'day'])
    if date_col:
        df['Fecha'] = pd.to_datetime(df[date_col], errors='coerce')
        df['Fecha'] = df['Fecha'].fillna(pd.Timestamp('2023-01-01'))
    else:
        df['Fecha'] = pd.date_range('2023-01-01', periods=len(df), freq='D')
    return df

In [ ]:
# Salud: crear Alto_Riesgo y ocupacion total por fecha
health_df = ensure_fecha(health_df)
risk_col = find_col(health_df, ['risk_level', 'risk level', 'risklevel'])
score_col = find_col(health_df, ['risk_score', 'overall_risk', 'score'])

if risk_col:
    health_df['Alto_Riesgo'] = health_df[risk_col].astype(str).str.lower().str.contains('high')
elif score_col:
    threshold = health_df[score_col].median()
    health_df['Alto_Riesgo'] = pd.to_numeric(health_df[score_col], errors='coerce') >= threshold
else:
    health_df['Alto_Riesgo'] = False

health_daily = health_df.groupby('Fecha').agg(
    Ocupacion_Total=('Alto_Riesgo', 'count')
,
,
,
,

: 
,
: {},
: null,
: [],
: [
,
,
,
,

In [ ]:
# Merge final de salud y stock
merged = stock_structured.merge(health_daily, on='Fecha', how='left')
merged_out = MERGED_DIR / 'Dataset_ALDIMI_Merged.csv'
merged.to_csv(merged_out, index=False)
print('Saved merged dataset:', merged_out)

corr = merged[['Consumo_Diario', 'Pacientes_Alto_Riesgo', 'Ocupacion_Total']].corr()
print('Correlation check:')
print(corr)

merged.head()